# 17 — Comparative Experiment (Repair vs. No-Repair)

**Maps to:** `report/Chapters/Task3.tex` §`T3:RepairCritique`.  
**Ticket:** TICKET-17.

Compare GA performance on kroA100 under two constraint-handling strategies:

1. **Repair**: naive (single-point) crossover produces infeasible offspring →
   repair mechanism restores feasibility by replacing duplicates with missing
   cities.
2. **No repair**: same naive crossover, but infeasible offspring survive as-is.
   Fitness is computed on the raw chromosome (visiting some cities twice and
   skipping others).

We deliberately use a **naive crossover** (not PMX/OX/CX) so that
infeasibility actually arises. With permutation-preserving operators, repair
is a no-op and the comparison is meaningless.

30 independent runs per strategy, followed by a Wilcoxon signed-rank test.

> **Phase 2 (after TICKET-13):** Extend to three arms by adding the
> penalty-function strategy from TICKET-13. See issue tracker for the
> re-open note.

---
## Setup

Import the experiment runner infrastructure from notebook 15.

In [1]:
%run ./15_experiment_runner.ipynb

Loaded kroA100: 100 cities


Best fitness : 84458.32
Known optimal: 21282
Gap          : 296.9%
Wall time    : 0.2s

Per-generation log (first 5 rows):
Saved: ../results/9e9a3301_seed0042.csv
Size : 11,657 bytes
Shape: (101, 16)
Cols : ['generation', 'best_fitness', 'mean_fitness', 'diversity', 'pop_size', 'n_generations', 'crossover_rate', 'mutation_rate', 'tournament_k', 'elitism_count', 'selection_method', 'crossover_method', 'mutation_method', 'repair_enabled', 'repair_strategy', 'seed']
Grid: 6 configurations
  976d5e2d | xover=pmx seed=1
  976d5e2d | xover=pmx seed=2
  976d5e2d | xover=pmx seed=3
  0fc01629 | xover=ox seed=1
  0fc01629 | xover=ox seed=2
  0fc01629 | xover=ox seed=3
Total: 6 | Completed: 6 | Pending: 0
Nothing to run — all results already exist.
Re-running the same grid (all should be skipped):
Total: 6 | Completed: 6 | Pending: 0
Nothing to run — all results already exist.


In [2]:
from scipy import stats
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

---
## Naive Single-Point Crossover

A simple single-point crossover that copies the first segment from parent A
and the second segment from parent B. This **does not** preserve the
permutation property — the child will typically contain duplicate cities and
miss others. This is the mechanism that makes repair necessary.

In [3]:
def naive_crossover(parent_a, parent_b, pt, _unused=None):
    child = np.empty_like(parent_a)
    child[:pt] = parent_a[:pt]
    child[pt:] = parent_b[pt:]
    return child

---
## Extended `run_experiment`

Override `run_experiment` to support `crossover_method="naive"`. The naive
crossover takes a single cut point rather than two.

In [4]:
def run_experiment(config, dist_matrix):
    n_cities = dist_matrix.shape[0]
    rng = np.random.default_rng(config.seed)

    xover_map = {"pmx": pmx, "ox": ox, "cx": cx, "naive": naive_crossover}
    xover_fn = xover_map[config.crossover_method]
    mutate_fn = {"swap": swap_mutation, "inversion": inversion_mutation}[config.mutation_method]

    population = random_population(config.pop_size, n_cities, rng)
    fitnesses = np.array([tour_length(ind, dist_matrix) for ind in population])

    records = []

    for gen in range(config.n_generations):
        diversity = pairwise_hamming(population)
        records.append({
            "generation": gen,
            "best_fitness": float(fitnesses.min()),
            "mean_fitness": float(fitnesses.mean()),
            "diversity": float(diversity),
        })

        new_pop = []

        elite_idx = np.argsort(fitnesses)[:config.elitism_count]
        for idx in elite_idx:
            new_pop.append(population[idx].copy())

        while len(new_pop) < config.pop_size:
            if config.selection_method == "tournament":
                p1 = tournament_select(population, fitnesses, config.tournament_k, rng)
                p2 = tournament_select(population, fitnesses, config.tournament_k, rng)
            else:
                p1 = roulette_select(population, fitnesses, rng)
                p2 = roulette_select(population, fitnesses, rng)

            if rng.random() < config.crossover_rate:
                if config.crossover_method == "cx":
                    child, _ = xover_fn(p1, p2)
                elif config.crossover_method == "naive":
                    pt = rng.integers(1, n_cities)
                    child = xover_fn(p1, p2, pt)
                else:
                    pts = sorted(rng.choice(n_cities, size=2, replace=False))
                    child = xover_fn(p1, p2, pts[0], pts[1])
            else:
                child = p1.copy()

            child = mutate_fn(child, config.mutation_rate, rng)

            if config.repair_enabled:
                child = repair(child, n_cities, dist_matrix=dist_matrix,
                               strategy=config.repair_strategy, rng=rng)

            new_pop.append(child)

        population = np.array(new_pop[:config.pop_size])
        fitnesses = np.array([tour_length(ind, dist_matrix) for ind in population])

    diversity = pairwise_hamming(population)
    records.append({
        "generation": config.n_generations,
        "best_fitness": float(fitnesses.min()),
        "mean_fitness": float(fitnesses.mean()),
        "diversity": float(diversity),
    })

    best_idx = np.argmin(fitnesses)
    return {
        "log": pd.DataFrame(records),
        "best_tour": population[best_idx],
        "best_fitness": float(fitnesses[best_idx]),
    }

---
## Load Benchmark

In [5]:
coords, dist_matrix = load_tsp(Path("../data/TSP-dataset/kroA100.tsp"))
n_cities = dist_matrix.shape[0]
print(f"Loaded kroA100: {n_cities} cities")
OPTIMAL = 21282

Loaded kroA100: 100 cities


---
## Experiment Design

Both arms use **naive crossover** so that infeasibility arises naturally.
The only difference is whether the repair mechanism is applied after
crossover + mutation.

- **repair**: `repair_enabled=True`, random insertion strategy.
- **no_repair**: `repair_enabled=False`, infeasible chromosomes survive.

The base configuration uses reasonable defaults. These will be replaced with
the best configuration from TICKET-16's parameter sweep once available.

Each strategy is run with 30 independent seeds for statistical power.

In [6]:
N_SEEDS = 30
SEEDS = list(range(1, N_SEEDS + 1))

BASE_CONFIG = {
    "pop_size": 100,
    "n_generations": 500,
    "crossover_rate": 0.8,
    "mutation_rate": 0.2,
    "tournament_k": 3,
    "elitism_count": 2,
    "selection_method": "tournament",
    "crossover_method": "naive",
    "mutation_method": "swap",
}

strategies = {
    "repair": {"repair_enabled": True, "repair_strategy": "random"},
    "no_repair": {"repair_enabled": False, "repair_strategy": "random"},
}

configs = {}
for name, overrides in strategies.items():
    configs[name] = build_grid(
        param_grid={k: [v] for k, v in overrides.items()},
        seeds=SEEDS,
        base=BASE_CONFIG,
    )
    print(f"{name:>12s}: {len(configs[name]):3d} configs  "
          f"hash={config_hash(configs[name][0])}")

all_configs = [c for group in configs.values() for c in group]
print(f"\nTotal: {len(all_configs)} runs")

      repair:  30 configs  hash=9d75f11b
   no_repair:  30 configs  hash=6ec6908d

Total: 60 runs


---
## Sanity Check — Naive Crossover Produces Infeasibility

Verify that the naive crossover actually breaks the permutation property,
confirming that the repair mechanism has work to do.

In [7]:
rng = np.random.default_rng(0)
p1 = rng.permutation(n_cities)
p2 = rng.permutation(n_cities)
pt = 30

child = naive_crossover(p1, p2, pt)
dups, missing = diagnose(child, n_cities)
print(f"Parent A valid : {is_valid_permutation(p1, n_cities)}")
print(f"Parent B valid : {is_valid_permutation(p2, n_cities)}")
print(f"Child valid    : {is_valid_permutation(child, n_cities)}")
print(f"Duplicates     : {len(dups)} positions")
print(f"Missing cities : {len(missing)}")

repaired = repair(child, n_cities, dist_matrix=dist_matrix, strategy="random", rng=rng)
print(f"Repaired valid : {is_valid_permutation(repaired, n_cities)}")

Parent A valid : True
Parent B valid : True
Child valid    : False
Duplicates     : 23 positions
Missing cities : 23
Repaired valid : True


---
## Run Experiments

Execute all pending configurations. Completed runs are automatically skipped
(resumability from notebook 15).

In [8]:
run_grid(all_configs, dist_matrix, n_workers=1)

Total: 60 | Completed: 0 | Pending: 60


  [1/60] 9d75f11b_seed0001.csv  best=47582  (2s elapsed, ~118s remaining)


  [2/60] 9d75f11b_seed0002.csv  best=51082  (4s elapsed, ~117s remaining)


  [3/60] 9d75f11b_seed0003.csv  best=53070  (6s elapsed, ~115s remaining)


  [4/60] 9d75f11b_seed0004.csv  best=43891  (8s elapsed, ~113s remaining)


...

  [57/60] 6ec6908d_seed0027.csv  best=39895  (98s elapsed, ~5s remaining)


  [58/60] 6ec6908d_seed0028.csv  best=32477  (100s elapsed, ~3s remaining)


  [59/60] 6ec6908d_seed0029.csv  best=44657  (101s elapsed, ~2s remaining)


  [60/60] 6ec6908d_seed0030.csv  best=34392  (102s elapsed, ~0s remaining)

Done: 60 runs in 102.4s


[PosixPath('../results/9d75f11b_seed0001.csv'),
 PosixPath('../results/9d75f11b_seed0002.csv'),
 PosixPath('../results/9d75f11b_seed0003.csv'),
 PosixPath('../results/9d75f11b_seed0004.csv'),
 PosixPath('../results/9d75f11b_seed0005.csv'),
 PosixPath('../results/9d75f11b_seed0006.csv'),
 PosixPath('../results/9d75f11b_seed0007.csv'),
 PosixPath('../results/9d75f11b_seed0008.csv'),
 PosixPath('../results/9d75f11b_seed0009.csv'),
 PosixPath('../results/9d75f11b_seed0010.csv'),
 PosixPath('../results/9d75f11b_seed0011.csv'),
 PosixPath('../results/9d75f11b_seed0012.csv'),
 PosixPath('../results/9d75f11b_seed0013.csv'),
 PosixPath('../results/9d75f11b_seed0014.csv'),
 PosixPath('../results/9d75f11b_seed0015.csv'),
 PosixPath('../results/9d75f11b_seed0016.csv'),
 PosixPath('../results/9d75f11b_seed0017.csv'),
 PosixPath('../results/9d75f11b_seed0018.csv'),
 PosixPath('../results/9d75f11b_seed0019.csv'),
 PosixPath('../results/9d75f11b_seed0020.csv'),
 PosixPath('../results/9d75f11b_seed0021

---
## Collect Results

Load all CSV results and extract final-generation metrics per strategy.

In [9]:
def collect_results(config_list, label):
    rows = []
    for c in config_list:
        path = result_path(c)
        if not path.exists():
            continue
        df = pd.read_csv(path)
        final = df.iloc[-1]
        rows.append({
            "strategy": label,
            "seed": c.seed,
            "best_fitness": final["best_fitness"],
            "mean_fitness": final["mean_fitness"],
            "diversity": final["diversity"],
        })
    return pd.DataFrame(rows)

results = pd.concat([
    collect_results(configs["repair"], "repair"),
    collect_results(configs["no_repair"], "no_repair"),
], ignore_index=True)

print(f"Collected {len(results)} results")
results.head()

Collected 60 results


,strategy,seed,best_fitness,mean_fitness,diversity
0,repair,1,47581.658732,48458.636263,0.025810
1,repair,2,51081.930352,51662.737968,0.005996
2,repair,3,53069.579405,54301.331917,0.015774
3,repair,4,43890.510061,44574.932805,0.015903
4,repair,5,50610.131039,51446.270729,0.016248


---
## Summary Statistics

Per-strategy summary: mean, std, min, median, max of best fitness across
the 30 runs. Gap is relative to the known optimal (21,282).

In [10]:
summary = results.groupby("strategy")["best_fitness"].agg(
    ["count", "mean", "std", "min", "median", "max"]
).round(2)

summary["gap_%"] = ((summary["mean"] - OPTIMAL) / OPTIMAL * 100).round(1)

print("Summary Statistics — Best Fitness (lower is better)")
print("=" * 75)
print(summary.to_string())

Summary Statistics — Best Fitness (lower is better)
           count      mean      std       min    median       max  gap_%
strategy                                                                
no_repair     30  38129.27  4135.41  28805.71  37750.48  47741.29   79.2
repair        30  49777.84  3186.86  43890.51  49406.18  56629.20  133.9


---
## Statistical Significance — Wilcoxon Signed-Rank Test

Paired test: for each seed $s$, we have `repair(s)` and `no_repair(s)`.
The Wilcoxon signed-rank test checks whether the median difference is
significantly different from zero.

- **H₀:** No difference in best fitness between repair and no-repair.
- **H₁:** There is a significant difference (two-sided).
- Significance level: α = 0.05.

In [11]:
repair_vals = results[results["strategy"] == "repair"].sort_values("seed")["best_fitness"].values
no_repair_vals = results[results["strategy"] == "no_repair"].sort_values("seed")["best_fitness"].values

stat, p_value = stats.wilcoxon(repair_vals, no_repair_vals)

print("Wilcoxon Signed-Rank Test")
print("=" * 40)
print(f"Test statistic : {stat:.2f}")
print(f"p-value        : {p_value:.6f}")
print(f"α              : 0.05")
print(f"Reject H₀      : {'Yes' if p_value < 0.05 else 'No'}")
print()

if p_value < 0.05:
    better = "repair" if repair_vals.mean() < no_repair_vals.mean() else "no_repair"
    print(f"Conclusion: statistically significant difference; "
          f"'{better}' produces lower (better) fitness.")
else:
    print("Conclusion: no statistically significant difference at α=0.05.")

Wilcoxon Signed-Rank Test
Test statistic : 0.00
p-value        : 0.000000
α              : 0.05
Reject H₀      : Yes

Conclusion: statistically significant difference; 'no_repair' produces lower (better) fitness.


### Interpretation

The result that **no-repair produces lower fitness** is expected and
illustrative — it is the core pathology that motivates repair mechanisms.

With naive crossover, offspring contain ~23 duplicate cities and ~23 missing
cities (out of 100). The `tour_length` function sums distances along the
chromosome as-is, so an infeasible chromosome visits a smaller subset of
cities (some twice, others never). Shorter tours over fewer unique cities
naturally yield lower distance sums — but these are **not valid TSP
solutions**.

The repair arm produces genuinely feasible tours (all 100 cities visited
exactly once), so its fitness values are higher but **meaningful**. This
is exactly the selection-pressure distortion described in TICKET-11:
infeasible chromosomes with artificially low fitness dominate the
population, steering the search away from the feasible region.

This finding directly supports the report's argument for why constraint
handling is necessary in GA for TSP.

---
## Convergence Comparison

Mean convergence curve (±1 std) across 30 seeds for each strategy.

In [12]:
def load_convergence(config_list, label):
    frames = []
    for c in config_list:
        path = result_path(c)
        if not path.exists():
            continue
        df = pd.read_csv(path)[["generation", "best_fitness"]].copy()
        df["seed"] = c.seed
        df["strategy"] = label
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

conv = pd.concat([
    load_convergence(configs["repair"], "repair"),
    load_convergence(configs["no_repair"], "no_repair"),
], ignore_index=True)

conv_summary = conv.groupby(["strategy", "generation"])["best_fitness"].agg(
    ["mean", "std"]).reset_index()

print("Convergence data shape:", conv_summary.shape)
conv_summary.head()

Convergence data shape: (1002, 4)


,strategy,generation,mean,std
0,no_repair,0,151041.605836,2603.788818
1,no_repair,1,146020.228679,3350.564503
2,no_repair,2,141704.006361,3100.810389
3,no_repair,3,137758.771529,4451.874427
4,no_repair,4,133698.670606,4275.476923


---
## Diversity Comparison

Mean diversity per generation for each strategy.

In [13]:
def load_diversity(config_list, label):
    frames = []
    for c in config_list:
        path = result_path(c)
        if not path.exists():
            continue
        df = pd.read_csv(path)[["generation", "diversity"]].copy()
        df["seed"] = c.seed
        df["strategy"] = label
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

div_data = pd.concat([
    load_diversity(configs["repair"], "repair"),
    load_diversity(configs["no_repair"], "no_repair"),
], ignore_index=True)

div_summary = div_data.groupby(["strategy", "generation"])["diversity"].agg(
    ["mean", "std"]).reset_index()

print("Diversity data shape:", div_summary.shape)
div_summary.head()

Diversity data shape: (1002, 4)


,strategy,generation,mean,std
0,no_repair,0,0.990026,0.000118
1,no_repair,1,0.972409,0.001896
2,no_repair,2,0.937177,0.009762
3,no_repair,3,0.880273,0.030637
4,no_repair,4,0.804833,0.061914


---
## Export Comparison CSV

Save the per-run summary to `results/compare.csv` for downstream plotting
notebooks (TICKET-18, TICKET-19).

In [14]:
compare_path = Path("../results/compare.csv")
results.to_csv(compare_path, index=False)
print(f"Saved: {compare_path}")
print(f"Shape: {results.shape}")
results.describe()

Saved: ../results/compare.csv
Shape: (60, 5)


,seed,best_fitness,mean_fitness,diversity
count,60.000000,60.000000,60.000000,60.000000
mean,15.500000,43953.556143,44854.922662,0.011404
std,8.728484,6920.631388,7061.880580,0.004415
min,1.000000,28805.712756,29247.118526,0.005198
25%,8.000000,37751.949093,38357.173236,0.007913
50%,15.500000,44910.530768,45998.213506,0.010462
75%,23.000000,49194.591445,50331.366575,0.014779
max,30.000000,56629.203467,57661.467799,0.025810


---
## Summary

This notebook compares two constraint-handling strategies on kroA100 using
**naive single-point crossover** (which produces infeasible offspring):

1. **Repair** (`repair_enabled=True`): duplicate cities are replaced with
   missing ones via random insertion after each crossover + mutation step.
2. **No repair** (`repair_enabled=False`): infeasible offspring survive as-is.
   Fitness is computed on the raw chromosome, which visits some cities twice
   and skips others.

30 independent runs per strategy, paired Wilcoxon signed-rank test for
statistical significance.

> **Phase 2 note:** Once TICKET-13 (penalty function) is implemented, this
> notebook will be extended to a three-arm comparison (repair vs. penalty
> vs. no-repair). The experiment design, statistical tests, and CSV schema
> are structured to accommodate the third arm with minimal changes.